In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import torch

print(f"CUDA_VISIBLE_DEVICES: {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"Available GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    current_device = torch.cuda.current_device()
    print(f"Current GPU Device: {current_device} (re-indexed by CUDA_VISIBLE_DEVICES)")
    print(f"Device Name: {torch.cuda.get_device_name(current_device)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(current_device) / 1024**2:.2f} MB")
    print(f"Memory Cached: {torch.cuda.memory_reserved(current_device) / 1024**2:.2f} MB")
else:
    print("No GPU available. Using CPU instead.")

CUDA_VISIBLE_DEVICES: 3
Available GPUs: 1
Current GPU Device: 0 (re-indexed by CUDA_VISIBLE_DEVICES)
Device Name: Tesla V100-DGXS-32GB
Memory Allocated: 0.00 MB
Memory Cached: 0.00 MB


In [39]:
#import Libraries

import zipfile
from io import BytesIO
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.data import HeteroData

In [40]:
# Load edgelists

# tweet_discusses_claim
TC_tensor = torch.load('../data/edgelists/TC_tensor.pt')
CT_tensor = torch.load('../data/edgelists/CT_tensor.pt')

# reply_reply_to_tweet
TR_r_tensor = torch.load('../data/edgelists/TR_r_tensor.pt')
RT_r_tensor = torch.load('../data/edgelists/RT_r_tensor.pt')

# reply_quote_of_tweet
TR_q_tensor = torch.load('../data/edgelists/TR_q_tensor.pt')
RT_q_tensor = torch.load('../data/edgelists/RT_q_tensor.pt')

# tweet_has_hashtag_hashtag
TH_tensor = torch.load('../data/edgelists/TH_tensor.pt')
HT_tensor = torch.load('../data/edgelists/HT_tensor.pt')

# tweet_has_article_article
TA_tensor = torch.load('../data/edgelists/TA_tensor.pt')
AT_tensor = torch.load('../data/edgelists/AT_tensor.pt')

# tweet_has_image_image
TI_tensor = torch.load('../data/edgelists/TI_tensor.pt')
IT_tensor = torch.load('../data/edgelists/IT_tensor.pt')

# tweet_mentions_user
TU_m_tensor = torch.load('../data/edgelists/TU_m_tensor.pt')
UT_m_tensor = torch.load('../data/edgelists/UT_m_tensor.pt')

# user_posted_tweet
UT_tensor = torch.load('../data/edgelists/UT_tensor.pt')
TU_tensor = torch.load('../data/edgelists/TU_tensor.pt')

# user_posted_reply
UR_tensor = torch.load('../data/edgelists/UR_tensor.pt')
RU_tensor = torch.load('../data/edgelists/RU_tensor.pt')

# user_retweeted_tweet
UT_r_tensor = torch.load('../data/edgelists/UT_r_tensor.pt')
TU_r_tensor = torch.load('../data/edgelists/TU_r_tensor.pt')

# user_follows_user
UU_f_tensor = torch.load('../data/edgelists/UU_f_tensor.pt')
UU_f_rev_tensor = torch.load('../data/edgelists/UU_f_rev_tensor.pt')

# user_mentions_user
UU_m_tensor = torch.load('../data/edgelists/UU_m_tensor.pt')
UU_m_rev_tensor = torch.load('../data/edgelists/UU_m_rev_tensor.pt')

# user_has_hashtag_hashtag
UH_tensor = torch.load('../data/edgelists/UH_tensor.pt')
HU_tensor = torch.load('../data/edgelists/HU_tensor.pt')

In [41]:
#Label Encoder for ground truth
from sklearn.preprocessing import LabelEncoder

def label_encoding(df, col):
    label_encoder = LabelEncoder()
    encoded_data = label_encoder.fit_transform(df[col])
    # original_data = label_encoder.inverse_transform(encoded_data)
    #print(original_data)
    return encoded_data


df_C = pd.read_parquet('/mnt/nas/pisani/dataset_mumin/claim_LP_split_60-15-25.parquet')


             



Data_0 --> only edges in the graphs and identity matrix

In [31]:
import scipy.sparse


def read_pickle_from_zip_folder(fname):
    fname_base = '/mnt/nas/pisani/dataset_mumin/mumin-small.v0.zip' #'D:\\CNR\\mumin-small.v0\\' 

    # Open the zip file
    with zipfile.ZipFile(fname_base, 'r') as zip_ref:
        
        with zip_ref.open(fname) as file:
            # Read the file content into a pandas dataframe
            return pd.read_pickle(BytesIO(file.read()))
        

def get_sparse_eye(size):
    eye = scipy.sparse.eye(size)
    coo = eye.tocoo()
    values = coo.data
    indices = torch.LongTensor([coo.row, coo.col])
    i = torch.sparse.FloatTensor(indices, torch.FloatTensor(values), torch.Size([size, size]))
    return i


#df_C = read_pickle_from_zip_folder('claim.pickle') --> nuovo file con mask
df_T = read_pickle_from_zip_folder('tweet.pickle')
df_R = read_pickle_from_zip_folder('reply.pickle')
df_U = read_pickle_from_zip_folder('user.pickle')
df_H = read_pickle_from_zip_folder('hashtag.pickle')
df_A = read_pickle_from_zip_folder('article.pickle')
df_I = read_pickle_from_zip_folder('image.pickle')

claim_identity = get_sparse_eye(df_C.shape[0])
tweet_identity = get_sparse_eye(df_T.shape[0])
reply_identity = get_sparse_eye(df_R.shape[0])
user_identity = get_sparse_eye(df_U.shape[0])
hashtag_identity = get_sparse_eye(df_H.shape[0])
article_identity = get_sparse_eye(df_A.shape[0])
image_identity = get_sparse_eye(df_I.shape[0])

# Example of converting sparse matrix back to dense if needed
# claim_identity_dense = claim_identity.to_dense()

In [32]:
# Initialize HeteroData object
data = HeteroData()

# NODES

data['claim'].x = claim_identity 
data['claim'].y = torch.tensor(label_encoding(df_C, 'label').tolist())
data['tweet'].x = tweet_identity 
data['reply'].x = reply_identity 
data['user'].x = user_identity 
data['hashtag'].x = hashtag_identity 
data['article'].x = article_identity 
data['image'].x = image_identity 


# EDGES

data['tweet', 'discusses', 'claim'].edge_index = TC_tensor
data['claim', 'is_discussed_by', 'tweet'].edge_index = CT_tensor

data['reply', 'reply_to', 'tweet'].edge_index = TR_r_tensor
data['tweet', 'is_replied_by', 'reply'].edge_index = RT_r_tensor

data['reply', 'quote_of', 'tweet'].edge_index = TR_q_tensor
data['tweet', 'is_quoted_by', 'reply'].edge_index = RT_q_tensor

data['tweet', 'has_hashtag', 'hashtag'].edge_index = TH_tensor
data['hashtag', 'is_hashtag_of', 'tweet'].edge_index = HT_tensor

data['tweet', 'has_article', 'article'].edge_index = TA_tensor
data['article', 'is_article_of', 'tweet'].edge_index = AT_tensor

data['tweet', 'has_image', 'image'].edge_index = TI_tensor
data['image', 'is_image_of', 'tweet'].edge_index = IT_tensor

data['tweet', 'mentions', 'user'].edge_index = TU_m_tensor
data['user', 'is_mentioned_by', 'tweet'].edge_index = UT_m_tensor

data['user', 'posted', 'tweet'].edge_index = UT_tensor
data['tweet', 'is_posted_by', 'user'].edge_index = TU_tensor

data['user', 'posted', 'reply'].edge_index = UR_tensor
data['reply', 'is_posted_by', 'user'].edge_index = RU_tensor

data['user', 'retweeted', 'tweet'].edge_index = UT_r_tensor
data['tweet', 'is_retweeted_by', 'user'].edge_index = TU_r_tensor

data['user', 'follows', 'user'].edge_index = UU_f_tensor
data['user', 'is_followed_by', 'user'].edge_index = UU_f_rev_tensor

data['user', 'mentions', 'user'].edge_index = UU_m_tensor
data['user', 'is_mentioned_by', 'user'].edge_index = UU_m_rev_tensor

data['user', 'has_hashtag', 'hashtag'].edge_index = UH_tensor
data['hashtag', 'is_hashtag_of', 'user'].edge_index = HU_tensor


Data_1 --> edges in the graph + meta-paths

In [21]:
from torch_geometric.transforms import AddMetaPaths

#Add metapaths # CTUTC, CTHTC

metapaths = [[('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'is_posted_by', 'user'), 
              ('user', 'posted', 'tweet'), 
              ('tweet', 'discusses', 'claim')], #CTUTC
             [('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'has_hashtag', 'hashtag'), 
              ('hashtag', 'is_hashtag_of', 'tweet'), 
              ('tweet', 'discusses', 'claim')], #CTHTC
               [('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'is_replied_by', 'reply'), 
              ('reply', 'reply_to', 'tweet'), 
              ('tweet', 'discusses', 'claim')], #CTRTC_r
              [('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'is_quoted_by', 'reply'), 
              ('reply', 'quote_of', 'tweet'), 
              ('tweet', 'discusses', 'claim')]] #CTRTC_q


data = AddMetaPaths(metapaths, weighted=True)(data)

In [11]:
data

HeteroData(
  metapath_dict={
    (claim, metapath_0, claim)=[4],
    (claim, metapath_1, claim)=[4],
    (claim, metapath_2, claim)=[4],
    (claim, metapath_3, claim)=[4],
  },
  claim={
    x=[2168, 2168],
    y=[2168],
  },
  tweet={ x=[4340, 4340] },
  reply={ x=[195459, 195459] },
  user={ x=[153168, 153168] },
  hashtag={ x=[28091, 28091] },
  article={ x=[1457, 1457] },
  image={ x=[1020, 1020] },
  (tweet, discusses, claim)={ edge_index=[2, 5081] },
  (claim, is_discussed_by, tweet)={ edge_index=[2, 5081] },
  (reply, reply_to, tweet)={ edge_index=[2, 90196] },
  (tweet, is_replied_by, reply)={ edge_index=[2, 90196] },
  (reply, quote_of, tweet)={ edge_index=[2, 101216] },
  (tweet, is_quoted_by, reply)={ edge_index=[2, 101216] },
  (tweet, has_hashtag, hashtag)={ edge_index=[2, 2289] },
  (hashtag, is_hashtag_of, tweet)={ edge_index=[2, 2289] },
  (tweet, has_article, article)={ edge_index=[2, 1898] },
  (article, is_article_of, tweet)={ edge_index=[2, 1898] },
  (tweet, has_

Data_2 --> Heterodata object with nodes/edges + meta-paths + node attributes

In [43]:
# Load node features

dim = 256

#CX = torch.load('../data/features/CX_'+str(dim)+'_tensor.pt')
CX = torch.load('../data/features/CX_'+str(dim)+'_tensor_v2_0_no_val.pt')
CY = torch.load('../data/features/CY_tensor.pt') 
TX = torch.load('../data/features/TX_'+str(dim)+'_tensor.pt') 
RX = torch.load('../data/features/RX_'+str(dim)+'_tensor.pt')
UX = torch.load('../data/features/UX_'+str(dim)+'_tensor.pt')  
HX = torch.load('../data/features/HX_'+str(dim)+'_tensor_v2.pt') #+ clusters
AX = torch.load('../data/features/AX_'+str(dim)+'_tensor.pt') 
IX = torch.load('../data/features/IX_'+str(dim)+'_tensor.pt') 

'''
#CX = torch.load('../data/features/CX_tensor.pt')
CX = torch.load('../data/features/CX_tensor_v2_0.pt')
CY = torch.load('../data/features/CY_tensor.pt') 
TX = torch.load('../data/features/TX_tensor.pt') 
RX = torch.load('../data/features/RX_tensor.pt')
UX = torch.load('../data/features/UX_tensor.pt')  
HX = torch.load('../data/features/HX_tensor_v2.pt') #+ clusters
AX = torch.load('../data/features/AX_tensor.pt') 
IX = torch.load('../data/features/IX_tensor_v2.pt') 
'''


In [44]:
# Initialize HeteroData object
data = HeteroData()

# NODES

data['claim'].x = CX #torch.tensor(pca_claim.transform(CX)).float()
data['claim'].y = CY
data['tweet'].x = TX #torch.tensor(pca_tweet.transform(TX)).float()
data['reply'].x = RX #torch.tensor(pca_reply.transform(RX)).float()
data['user'].x =  UX #torch.tensor(pca_user.transform(UX)).float()
data['hashtag'].x = HX #torch.tensor(pca_hashtag.transform(HX)).float()
data['article'].x = AX #torch.tensor(pca_article.transform(AX)).float()
data['image'].x = IX #torch.tensor(pca_image.transform(IX)).float()


# EDGES

data['tweet', 'discusses', 'claim'].edge_index = TC_tensor
data['claim', 'is_discussed_by', 'tweet'].edge_index = CT_tensor

data['reply', 'reply_to', 'tweet'].edge_index = TR_r_tensor
data['tweet', 'is_replied_by', 'reply'].edge_index = RT_r_tensor

data['reply', 'quote_of', 'tweet'].edge_index = TR_q_tensor
data['tweet', 'is_quoted_by', 'reply'].edge_index = RT_q_tensor

data['tweet', 'has_hashtag', 'hashtag'].edge_index = TH_tensor
data['hashtag', 'is_hashtag_of', 'tweet'].edge_index = HT_tensor

data['tweet', 'has_article', 'article'].edge_index = TA_tensor
data['article', 'is_article_of', 'tweet'].edge_index = AT_tensor

data['tweet', 'has_image', 'image'].edge_index = TI_tensor
data['image', 'is_image_of', 'tweet'].edge_index = IT_tensor

data['tweet', 'mentions', 'user'].edge_index = TU_m_tensor
data['user', 'is_mentioned_by', 'tweet'].edge_index = UT_m_tensor

data['user', 'posted', 'tweet'].edge_index = UT_tensor
data['tweet', 'is_posted_by', 'user'].edge_index = TU_tensor

data['user', 'posted', 'reply'].edge_index = UR_tensor
data['reply', 'is_posted_by', 'user'].edge_index = RU_tensor

data['user', 'retweeted', 'tweet'].edge_index = UT_r_tensor
data['tweet', 'is_retweeted_by', 'user'].edge_index = TU_r_tensor

data['user', 'follows', 'user'].edge_index = UU_f_tensor
data['user', 'is_followed_by', 'user'].edge_index = UU_f_rev_tensor

data['user', 'mentions', 'user'].edge_index = UU_m_tensor
data['user', 'is_mentioned_by', 'user'].edge_index = UU_m_rev_tensor

data['user', 'has_hashtag', 'hashtag'].edge_index = UH_tensor
data['hashtag', 'is_hashtag_of', 'user'].edge_index = HU_tensor

In [45]:
from torch_geometric.transforms import AddMetaPaths

#Add metapaths # CTUTC, CTHTC

metapaths = [[('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'is_posted_by', 'user'), 
              ('user', 'posted', 'tweet'), 
              ('tweet', 'discusses', 'claim')], #CTUTC
             [('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'has_hashtag', 'hashtag'), 
              ('hashtag', 'is_hashtag_of', 'tweet'), 
              ('tweet', 'discusses', 'claim')], #CTHTC
               [('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'is_replied_by', 'reply'), 
              ('reply', 'reply_to', 'tweet'), 
              ('tweet', 'discusses', 'claim')], #CTRTC_r
              [('claim', 'is_discussed_by', 'tweet'), 
              ('tweet', 'is_quoted_by', 'reply'), 
              ('reply', 'quote_of', 'tweet'), 
              ('tweet', 'discusses', 'claim')]] #CTRTC_q


data = AddMetaPaths(metapaths, weighted=True)(data)

In [8]:
data

HeteroData(
  metapath_dict={
    (claim, metapath_0, claim)=[4],
    (claim, metapath_1, claim)=[4],
    (claim, metapath_2, claim)=[4],
    (claim, metapath_3, claim)=[4],
  },
  claim={
    x=[2168, 256],
    y=[2168],
  },
  tweet={ x=[4340, 256] },
  reply={ x=[195459, 256] },
  user={ x=[153168, 256] },
  hashtag={ x=[28091, 256] },
  article={ x=[1457, 256] },
  image={ x=[1020, 256] },
  (tweet, discusses, claim)={ edge_index=[2, 5081] },
  (claim, is_discussed_by, tweet)={ edge_index=[2, 5081] },
  (reply, reply_to, tweet)={ edge_index=[2, 90196] },
  (tweet, is_replied_by, reply)={ edge_index=[2, 90196] },
  (reply, quote_of, tweet)={ edge_index=[2, 101216] },
  (tweet, is_quoted_by, reply)={ edge_index=[2, 101216] },
  (tweet, has_hashtag, hashtag)={ edge_index=[2, 2289] },
  (hashtag, is_hashtag_of, tweet)={ edge_index=[2, 2289] },
  (tweet, has_article, article)={ edge_index=[2, 1898] },
  (article, is_article_of, tweet)={ edge_index=[2, 1898] },
  (tweet, has_image, image

In [46]:
def compute_weights():
    targets = data['claim'].y
    num_zeros = (targets == 0).sum().item()
    num_ones = (targets == 1).sum().item()
    total_samples = len(targets)

    # each weight inversely proportional to the class frequency
    weight_for_zeros = total_samples / num_zeros
    weight_for_ones = total_samples / num_ones

    # Normalization
    weight_sum = weight_for_zeros + weight_for_ones
    weight_for_zeros /= weight_sum
    weight_for_ones /= weight_sum

    weights = torch.tensor([weight_for_zeros, weight_for_ones])

    print(f"Number of 0s: {num_zeros}")
    print(f"Number of 1s: {num_ones}")
    print(f"Computed weights: {weights}")
    return weights

In [48]:
# Methods for statistics computation

from statistics import stdev

def process_metric(df, col):
    data = df[col]
    media = data.mean()
    st_dev = stdev(data.tolist())
    return media, st_dev

#df = pd.DataFrame(columns=['F1_micro', 'F1_macro', 'F1_weighted', 'ROC-AUC'])
def processing_results(df):
    res = pd.DataFrame(columns=df.columns) #columns=['F1_micro', 'F1_macro', 'F1_weighted', 'ROC-AUC']
    
    #Compute mean and standard deviation
    for col in df.columns:
        lista = df[col].tolist()
        media, st_dev = process_metric(df, col)
        stringa = "{:0.4f}".format(media)+u"\u00B1"+"{:0.4f}".format(st_dev)
        lista.append(stringa)
        res[col] = lista

    return res

In [49]:
# GAT module using to_hetero

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, Linear


class GAT(torch.nn.Module):
    def __init__(self, hidden_channels=128, out_channels=2, dropout=0, num_layers=2): #hidden_channels=64
        super().__init__()
        self.num_layers = num_layers
        
        self.convs = torch.nn.ModuleList()
        self.lins = torch.nn.ModuleList()

        # First layer
        self.convs.append(GATv2Conv((-1, -1), hidden_channels, add_self_loops=False, dropout=dropout))
        self.lins.append(Linear(-1, hidden_channels))

        # Intermediate layers
        for _ in range(num_layers - 2):
            self.convs.append(GATv2Conv((-1, -1), hidden_channels, add_self_loops=False, dropout=dropout))
            self.lins.append(Linear(-1, hidden_channels))

        # Last layer
        self.convs.append(GATv2Conv((-1, -1), hidden_channels, add_self_loops=False, dropout=dropout))  # Change out_channels to hidden_channels
        self.lins.append(Linear(-1, hidden_channels))  # Change out_channels to hidden_channels
        
        self.final_conv = GATv2Conv((-1, -1), out_channels, add_self_loops=False, dropout=dropout)  # Add final GAT layer for classification
        self.final_lin = Linear(-1, out_channels)  # Add final linear layer for classification

    
    def forward(self, x, edge_index):
        for i in range(self.num_layers - 1):
            x = self.convs[i](x, edge_index) + self.lins[i](x.relu())
            x = x.relu()

        # Save embeddings before the final layer
        self.embeddings = self.convs[-1](x, edge_index) + self.lins[-1](x.relu())

        # Last layer (no ReLU after the last convolution)
        x = self.final_conv(self.embeddings, edge_index) + self.final_lin(self.embeddings.relu())
        return x, self.embeddings


In [50]:
from torch_geometric.nn import to_hetero
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
import time

# Model training    

def train_node_classifier(model, data, optimizer, criterion, n_epochs=200): 

    for epoch in range(1, n_epochs + 1):
        model.train()
        optimizer.zero_grad()
        out, _ = model(data.x_dict, data.edge_index_dict)
        mask = data['claim'].train_mask
        loss = criterion(out['claim'][mask], data['claim'].y[mask]) 
        loss.backward()
        optimizer.step()

        pred = out['claim'].argmax(dim=1) ## Use the class with highest probability. 
       
        f1_micro, f1_macro, f1_weigh, auc, precision_0, recall_0, precision_1, recall_1 = eval_node_classifier(model, data, run) 

        if epoch % 20 == 0:
            print(f'Epoch: {epoch:03d}, Train Loss: {loss:.3f}, Val f1_micro: {f1_micro:.3f}, Val f1_macro: {f1_macro:.3f}')

    return model


def eval_node_classifier(model, data, run):

    model.eval()
    with torch.no_grad():
        #pred = model(data.x_dict, data.edge_index_dict)['claim'].argmax(dim=-1)
        out, embeddings = model(data.x_dict, data.edge_index_dict)
        pred = out['claim'].argmax(dim=-1)
        mask = data['claim'].val_mask
        #correct = (pred[mask] == data['claim'].y[mask]).sum()
        f1_micro = f1_score(data['claim'].y.cpu(), pred.cpu(), average='micro')
        f1_macro = f1_score(data['claim'].y.cpu(), pred.cpu(), average='macro')
        f1_weigh = f1_score(data['claim'].y.cpu(), pred.cpu(), average='weighted')
        auc = roc_auc_score(data['claim'].y.cpu(), pred.cpu(), average='weighted')
        prec_0 = precision_score(data['claim'].y.cpu(), pred.cpu(), pos_label=0, average='binary')
        rec_0 = recall_score(data['claim'].y.cpu(), pred.cpu(), pos_label=0, average='binary')
        prec_1 = precision_score(data['claim'].y.cpu(), pred.cpu(), pos_label=1, average='binary')
        rec_1 = recall_score(data['claim'].y.cpu(), pred.cpu(), pos_label=1, average='binary')

        # Save embeddings for validation set
        val_embeddings = embeddings['claim'].cpu().numpy()
        np.save('../data/embeddings/val_15/test_CR_featsall_embeddings_'+str(run)+'.npy', val_embeddings)

        return f1_micro, f1_macro, f1_weigh, auc, prec_0, rec_0, prec_1, rec_1
  
  
num_run = 5 

models = []

l_micro = []
l_macro = []
l_weigh = []
l_auc = []
l_prec_0 = []
l_rec_0 = []
l_prec_1 = []
l_rec_1 = []

for run in range(num_run):
    print('##### RUN '+str((run+1))+' #####')
    start_time = time.time()

    # SET THE MODEL
    model = None
    model = GAT(hidden_channels=64, out_channels=2, dropout=0.4, num_layers=3)
    model = to_hetero(model, data.metadata(), aggr='sum')

    # SET THE DATA
    # TRAIN - VAL - TEST SPLIT

    #split 60-15-25
    
    data['claim'].train_mask = torch.tensor(df_C['train_mask_LP_'+str(run)].values, dtype=torch.bool)
    data['claim'].val_mask = torch.tensor(df_C['val_mask_LP_'+str(run)].values, dtype=torch.bool)
    data['claim'].test_mask = torch.tensor(df_C['test_mask_LP_'+str(run)].values, dtype=torch.bool)
    '''
    #split 75-0-25
    data['claim'].train_mask = torch.tensor(df_C['train_mask_LP_'+str(run)].values, dtype=torch.bool)
    data['claim'].val_mask = torch.tensor(df_C['val_mask_LP_'+str(run)].values, dtype=torch.bool)
    data['claim'].test_mask = torch.tensor(df_C['test_mask_LP_'+str(run)].values, dtype=torch.bool)
    '''

    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    data, model = data.to(device), model.to(device)
    

    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4) #lr=0.0001
    weights = compute_weights().float().to(device)
    criterion = nn.CrossEntropyLoss(weights)

    model = train_node_classifier(model, data, optimizer, criterion, n_epochs=200) #200, 500, 800
    models.append(model)

    f1_micro, f1_macro, f1_weigh, auc, prec_0, rec_0, prec_1, rec_1 = eval_node_classifier(model, data, run) 
    print(f'f1-micro: {f1_micro:.3f}, f1-macro: {f1_macro:.3f}, f1-weighted: {f1_weigh:.3f}, roc-auc: {auc:.3f}')

    l_micro.append(f1_micro)
    l_macro.append(f1_macro)
    l_weigh.append(f1_weigh)
    l_auc.append(auc)
    l_prec_0.append(prec_0)
    l_rec_0.append(rec_0)
    l_prec_1.append(prec_1)
    l_rec_1.append(rec_1)

    end_time = time.time()
    execution_time = end_time-start_time
    print(f'Execution time: {execution_time} seconds')


# save best model
top = l_macro.index(max(l_macro))
torch.save(models[top].state_dict(), '../data/models/val_15/GAT_embeddings_featsall_test_CR_model.pth')


df = pd.DataFrame(columns=['F1_micro', 'F1_macro', 'F1_weighted', 'ROC-AUC', 'Precision_0', 'Recall_0', 'Precision_1', 'Recall_1'])
df['F1_micro'] = l_micro
df['F1_macro'] = l_macro
df['F1_weighted'] = l_weigh
df['ROC-AUC'] = l_auc
df['Precision_0'] = l_prec_0
df['Recall_0'] = l_rec_0
df['Precision_1'] = l_prec_1
df['Recall_1'] = l_rec_1
df_ok = processing_results(df)
df_ok.to_excel('../data/results/val_15/embeddings_featsall_test_CR_results.xlsx', index=False)


##### RUN 1 #####
Number of 0s: 158
Number of 1s: 2010
Computed weights: tensor([0.9271, 0.0729])
Epoch: 020, Train Loss: 734.989, Val f1_micro: 0.111, Val f1_macro: 0.110
Epoch: 040, Train Loss: 330.289, Val f1_micro: 0.301, Val f1_macro: 0.278
Epoch: 060, Train Loss: 194.825, Val f1_micro: 0.641, Val f1_macro: 0.494
Epoch: 080, Train Loss: 188.763, Val f1_micro: 0.746, Val f1_macro: 0.559
Epoch: 100, Train Loss: 101.607, Val f1_micro: 0.886, Val f1_macro: 0.604
Epoch: 120, Train Loss: 90.732, Val f1_micro: 0.792, Val f1_macro: 0.595
Epoch: 140, Train Loss: 82.811, Val f1_micro: 0.789, Val f1_macro: 0.595
Epoch: 160, Train Loss: 80.001, Val f1_micro: 0.843, Val f1_macro: 0.636
Epoch: 180, Train Loss: 66.691, Val f1_micro: 0.770, Val f1_macro: 0.582
Epoch: 200, Train Loss: 60.507, Val f1_micro: 0.804, Val f1_macro: 0.602
f1-micro: 0.804, f1-macro: 0.602, f1-weighted: 0.844, roc-auc: 0.725
Execution time: 177.57947611808777 seconds
##### RUN 2 #####
Number of 0s: 158
Number of 1s: 2010


/home/martirano/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/martirano/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch: 020, Train Loss: 412.082, Val f1_micro: 0.146, Val f1_macro: 0.146
Epoch: 040, Train Loss: 281.582, Val f1_micro: 0.328, Val f1_macro: 0.302
Epoch: 060, Train Loss: 156.143, Val f1_micro: 0.782, Val f1_macro: 0.554
Epoch: 080, Train Loss: 92.787, Val f1_micro: 0.540, Val f1_macro: 0.442
Epoch: 100, Train Loss: 103.909, Val f1_micro: 0.751, Val f1_macro: 0.515
Epoch: 120, Train Loss: 49.178, Val f1_micro: 0.610, Val f1_macro: 0.479
Epoch: 140, Train Loss: 45.909, Val f1_micro: 0.723, Val f1_macro: 0.545
Epoch: 160, Train Loss: 30.424, Val f1_micro: 0.693, Val f1_macro: 0.541
Epoch: 180, Train Loss: 39.443, Val f1_micro: 0.512, Val f1_macro: 0.423
Epoch: 200, Train Loss: 21.519, Val f1_micro: 0.847, Val f1_macro: 0.632
f1-micro: 0.847, f1-macro: 0.632, f1-weighted: 0.872, roc-auc: 0.719
Execution time: 216.25557565689087 seconds
##### RUN 4 #####
Number of 0s: 158
Number of 1s: 2010
Computed weights: tensor([0.9271, 0.0729])
Epoch: 020, Train Loss: 338.262, Val f1_micro: 0.843, Va

In [44]:
res = pd.read_excel('../data/results/val_15/GAT_results.xlsx')
res

,F1_micro,F1_macro,F1_weighted,ROC-AUC,Precision 0,Recall 0,Precision 1,Recall 1
0,0.964945,0.866388,0.964415,0.855715,0.777027,0.727848,0.978713,0.983582
1,0.963561,0.867109,0.963822,0.872464,0.742331,0.765823,0.981546,0.979104
2,0.958487,0.850741,0.959072,0.86098,0.702381,0.746835,0.98,0.975124
3,0.962177,0.863237,0.962606,0.871717,0.728916,0.765823,0.981518,0.977612
4,0.962638,0.854516,0.961655,0.836977,0.77305,0.689873,0.975826,0.98408
5,0.9624±0.0024,0.8604±0.0074,0.9623±0.0021,0.8596±0.0145,0.7447±0.0312,0.7392±0.0318,0.9795±0.0024,0.9799±0.0039


In [46]:
# Model loading

from torch_geometric.nn import to_hetero
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

def load_model(hidden_channels,out_channels,dropout,data,weigths_filename):
    model = GAT(hidden_channels, out_channels,dropout)
    model = to_hetero(model, data.metadata(), aggr='sum')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data, model = data.to(device), model.to(device)

    with torch.no_grad():
        model.eval()
        model(data.x_dict, data.edge_index_dict)
        model.train()
    model.load_state_dict(torch.load(weigths_filename))
    return model

In [48]:
# Alternative to training: load saved model  

def eval_node_classifier(model, data):

    model.eval()
    with torch.no_grad():
        pred = model(data.x_dict, data.edge_index_dict)['claim'].argmax(dim=-1)
        mask = data['claim'].val_mask

        f1_micro = f1_score(data['claim'].y.cpu(), pred.cpu(), average='micro')
        f1_macro = f1_score(data['claim'].y.cpu(), pred.cpu(), average='macro')
        f1_weigh = f1_score(data['claim'].y.cpu(), pred.cpu(), average='weighted')
        auc = roc_auc_score(data['claim'].y.cpu(), pred.cpu(), average='weighted')
        prec_0 = precision_score(data['claim'].y.cpu(), pred.cpu(), pos_label=0, average='binary')
        rec_0 = recall_score(data['claim'].y.cpu(), pred.cpu(), pos_label=0, average='binary')
        prec_1 = precision_score(data['claim'].y.cpu(), pred.cpu(), pos_label=1, average='binary')
        rec_1 = recall_score(data['claim'].y.cpu(), pred.cpu(), pos_label=1, average='binary')

        # Identify false negatives for class 0
        true_labels = data['claim'].y[mask]
        pred_labels = pred[mask]
        false_negatives_0 = (true_labels == 0) & (pred_labels != 0)
        false_negative_indices = torch.where(false_negatives_0)[0]

        return f1_micro, f1_macro, f1_weigh, auc, prec_0, rec_0, prec_1, rec_1, false_negative_indices.cpu().numpy()
    
    

# LOAD SAVED MODEL
model = None
model = load_model(64,2,0.4,data,'../data/models/val_15/GAT_no_feats_model.pth')

# CORRESPONDING DATA ### from results.xlsx index of top model (max f1-macro)
run = 1
data['claim'].train_mask = torch.tensor(df_C['train_mask_LP_'+str(run)].values, dtype=torch.bool)
data['claim'].val_mask = torch.tensor(df_C['val_mask_LP_'+str(run)].values, dtype=torch.bool)
data['claim'].test_mask = torch.tensor(df_C['test_mask_LP_'+str(run)].values, dtype=torch.bool)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data, model = data.to(device), model.to(device)

f1_micro, f1_macro, f1_weigh, auc, prec_0, rec_0, prec_1, rec_1, FP_indices = eval_node_classifier(model, data) 

#print(f'f1-micro: {f1_micro:.3f}, f1-macro: {f1_macro:.3f}, f1-weighted: {f1_weigh:.3f}, , roc-auc: {auc:.3f}')
print(FP_indices)

[ 50 174 182 212 239 263 264 266 296 297 300 329 334 348 350 352 353 354
 361 421 426]


In [50]:
# Find corresponding indices

def count_edges_per_node(data, false_negative_indices, relation):
    # Extract the relevant edge index for the specified relation
    edge_index = data[relation].edge_index
    
    # Initialize a dictionary to store the counts
    edge_counts = {idx.item(): 0 for idx in false_negative_indices}

    # Iterate through the edges and count the occurrences of each false negative index as source node
    for idx in false_negative_indices:
        idx = idx.item()
        # Count how many times `idx` appears in the source nodes (first row of edge_index)
        edge_counts[idx] = (edge_index[0] == idx).sum().item()

    return edge_counts


relation = ('claim', 'is_discussed_by', 'tweet')
edge_counts = count_edges_per_node(data, FP_indices, relation)

for idx, count in edge_counts.items():
    print(f"Index {idx} has {count} edges of type 'claim is discussed by tweet' as source node.")

Index 50 has 2 edges of type 'claim is discussed by tweet' as source node.
Index 174 has 1 edges of type 'claim is discussed by tweet' as source node.
Index 182 has 2 edges of type 'claim is discussed by tweet' as source node.
Index 212 has 2 edges of type 'claim is discussed by tweet' as source node.
Index 239 has 3 edges of type 'claim is discussed by tweet' as source node.
Index 263 has 1 edges of type 'claim is discussed by tweet' as source node.
Index 264 has 1 edges of type 'claim is discussed by tweet' as source node.
Index 266 has 3 edges of type 'claim is discussed by tweet' as source node.
Index 296 has 1 edges of type 'claim is discussed by tweet' as source node.
Index 297 has 1 edges of type 'claim is discussed by tweet' as source node.
Index 300 has 1 edges of type 'claim is discussed by tweet' as source node.
Index 329 has 2 edges of type 'claim is discussed by tweet' as source node.
Index 334 has 7 edges of type 'claim is discussed by tweet' as source node.
Index 348 has

In [54]:
def count_edges_and_metapaths(data, false_negative_indices, relation, metapaths):
    # Extract the relevant edge index for the specified relation
    edge_index = data[relation].edge_index
    
    # Initialize dictionaries to store counts
    edge_counts = {idx.item(): 0 for idx in false_negative_indices}
    metapath_counts = {idx.item(): {mp: 0 for mp in metapaths} for idx in false_negative_indices}
    
    # Count direct edges
    for idx in false_negative_indices:
        idx = idx.item()
        edge_counts[idx] = (edge_index[0] == idx).sum().item()
    
    # Count occurrences for each metapath
    for metapath in metapaths:
        mp_edge_index = data[metapath].edge_index
        for idx in false_negative_indices:
            idx = idx.item()
            # Count occurrences of `idx` in the source nodes (first row of edge_index) for the metapath
            metapath_counts[idx][metapath] = (mp_edge_index[0] == idx).sum().item()
    
    return edge_counts, metapath_counts

# Define the relation and metapaths
relation = ('claim', 'is_discussed_by', 'tweet')
metapaths = [
    ('claim', 'metapath_0', 'claim'),
    ('claim', 'metapath_1', 'claim'),
    ('claim', 'metapath_2', 'claim'),
    ('claim', 'metapath_3', 'claim')
]

# Get the edge counts and metapath counts for each false negative index
edge_counts, metapath_counts = count_edges_and_metapaths(data, FP_indices, relation, metapaths)

# Print the index, the number of edges of type "claim is discussed by tweet", and the metapath counts
for idx, count in edge_counts.items():
    print(f"Index {idx} has {count} edges of type '{relation[1]}' as source node.")
    for metapath, mp_count in metapath_counts[idx].items():
        print(f"  Metapath '{metapath}' count: {mp_count}")


Index 50 has 2 edges of type 'is_discussed_by' as source node.
  Metapath '('claim', 'metapath_0', 'claim')' count: 7
  Metapath '('claim', 'metapath_1', 'claim')' count: 12
  Metapath '('claim', 'metapath_2', 'claim')' count: 6
  Metapath '('claim', 'metapath_3', 'claim')' count: 2
Index 174 has 1 edges of type 'is_discussed_by' as source node.
  Metapath '('claim', 'metapath_0', 'claim')' count: 1
  Metapath '('claim', 'metapath_1', 'claim')' count: 0
  Metapath '('claim', 'metapath_2', 'claim')' count: 1
  Metapath '('claim', 'metapath_3', 'claim')' count: 1
Index 182 has 2 edges of type 'is_discussed_by' as source node.
  Metapath '('claim', 'metapath_0', 'claim')' count: 1
  Metapath '('claim', 'metapath_1', 'claim')' count: 0
  Metapath '('claim', 'metapath_2', 'claim')' count: 1
  Metapath '('claim', 'metapath_3', 'claim')' count: 1
Index 212 has 2 edges of type 'is_discussed_by' as source node.
  Metapath '('claim', 'metapath_0', 'claim')' count: 2
  Metapath '('claim', 'metapa

In [55]:
import pandas as pd

def count_edges_and_metapaths(data, false_negative_indices, relation, metapaths):
    # Extract the relevant edge index for the specified relation
    edge_index = data[relation].edge_index
    
    # Initialize dictionaries to store counts
    edge_counts = {idx.item(): 0 for idx in false_negative_indices}
    metapath_counts = {idx.item(): {mp: 0 for mp in metapaths} for idx in false_negative_indices}
    
    # Count direct edges
    for idx in false_negative_indices:
        idx = idx.item()
        edge_counts[idx] = (edge_index[0] == idx).sum().item()
    
    # Count occurrences for each metapath
    for metapath in metapaths:
        mp_edge_index = data[metapath].edge_index
        for idx in false_negative_indices:
            idx = idx.item()
            # Count occurrences of `idx` in the source nodes (first row of edge_index) for the metapath
            metapath_counts[idx][metapath] = (mp_edge_index[0] == idx).sum().item()
    
    return edge_counts, metapath_counts

def save_counts_to_dataframe(edge_counts, metapath_counts, metapaths, filename='counts.csv'):
    # Create lists to hold the rows for the DataFrame
    data_rows = []
    
    for idx, edge_count in edge_counts.items():
        # Create a row for the DataFrame
        row = [idx, edge_count] + [metapath_counts[idx][mp] for mp in metapaths]
        data_rows.append(row)
    
    # Create a DataFrame from the rows
    columns = ['Index', 'Edge Count'] + [f'Metapath {i} Count' for i in range(len(metapaths))]
    df = pd.DataFrame(data_rows, columns=columns)
    
    # Save the DataFrame to a CSV file
    df.to_csv(filename, index=False)
    print(f"Data saved to {filename}")

# Define the relation and metapaths
relation = ('claim', 'is_discussed_by', 'tweet')
metapaths = [
    ('claim', 'metapath_0', 'claim'),
    ('claim', 'metapath_1', 'claim'),
    ('claim', 'metapath_2', 'claim'),
    ('claim', 'metapath_3', 'claim')
]


# Get the edge counts and metapath counts for each false negative index
edge_counts, metapath_counts = count_edges_and_metapaths(data, FP_indices, relation, metapaths)

# Save the counts to a CSV file
save_counts_to_dataframe(edge_counts, metapath_counts, metapaths, 'counts.csv')


Data saved to counts.csv


In [56]:
df = pd.read_csv('counts.csv')
df

,Index,Edge Count,Metapath 0 Count,Metapath 1 Count,Metapath 2 Count,Metapath 3 Count
0,50,2,7,12,6,2
1,174,1,1,0,1,1
2,182,2,1,0,1,1
3,212,2,2,2,1,1
4,239,3,2,0,2,2
5,263,1,0,0,1,0
6,264,1,0,0,1,1
7,266,3,1,1,1,1
8,296,1,14,0,14,14
9,297,1,14,0,14,14
